### IMPORTS AND LOAD SAVED DATA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import joblib
import os 

hourly_power = pd.read_csv("../results/hourly_power.csv", index_col=0, parse_dates=True).squeeze()
anomalies = pd.read_csv("../results/anomalies.csv", index_col=0, parse_dates=True)

print(anomalies.shape)
print(anomalies['anomaly_type'].value_counts())


### FEATURE ENGINEERING

In [ ]:
def build_features(series):
    """
    Build time-series features for each data point.
    These capture local context that helps distinguish anomaly types
    """
    df = pd.DataFrame({'Value': series})

    # Rolling statistics
    for w in [3, 6, 24]:
        df[f"rolling_mean_{w}h"] = series.rolling(w, min_periods=1).mean()
        df[f"rolling_std_{w}h"] = series.rolling(w, min_periods=1).std().fillna(0)
        df[f"rolling_var_{w}h"] = series.rolling(w, min_periods=1).var().fillna(0)

    # Deviation from local mean
    df['dev_from_mean_6h'] = (series - df['rolling_mean_6h']).abs()
    df['dev_from_mean_24h'] = (series - df['rolling_mean_24h']).abs()

    # Lag features (what happened before/after)
    df['lag_1h'] = series.shift(1)
    df['lag_2h'] = series.shift(2)
    df['lead_1h'] = series.shift(-1)

    # Time of day features
    df['hour'] = series.index.hour
    df['day_of_week'] = series.index.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    # Fill Nans 
    df = df.fillna(df.median())
    return df.drop(columns=['Value'])

features = build_features(hourly_power)
print(f"Feature matrix shape: {features.shape}")
print(features.columns.tolist())


### PREPARE TRAINING DATA

In [ ]:
# TODO: Implement the rest of cells, understand intuition for choice of features, how binning was done since data is time-series. 


# Align features with annomaly labels
X = features.copy()
y = anomalies['label']

# Drop rows where index does not align 
X, y = X.align(y, join='inner', axis=0) # inner join means we only keep rows where both X and y have data. axis =0 means we are aligning along the index (rows)
print(f"Training set size: {X.shape}")
print(f"Label distribution:\n{y.value_counts()}")

# Train/test split - keep the time-series order (no shuffling)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Train set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")


### TRAIN XGBOOST 

In [ ]:
model = XGBClassifier(
    n_estimators=200, # number of trees to build
    max_depth = 6, # maximum depth of each tree, controls model complexity
    learning_rate = 0.1, # step size shrinkage used in update to prevent overfitting
    subsample = 0.8, # fraction of samples to be used for fitting the individual base learners, helps prevent overfitting. This means that each tree is trained on a random subset of 80% of the data.
    use_label_encoder = False, # disable the use of label encoder. If set to True, it will encode the labels with value between 0 and n_classes-1. Since we are using a multi-class classification problem, we can set this to False and directly use the original labels.
    eval_metric = 'mlogloss', # evaluation metric for multi-class classification. It stands for "multi-class logarithmic loss". It measures the performance of a classification model where the prediction input is a probability value between 0 and 1. The lower the log loss, the better the model's performance.
    random_state=42 # set a random seed for reproducibility. This means that every time you run the code, you will get the same results, which is important for debugging and comparing model performance.
)

model.fit(X_train, y_train, eval_set = [(X_test, y_test)], verbose = 50) # verbose = 50 means that it will print the evaluation metric every 50 iterations during training. This allows you to monitor the model's performance on the test set as it trains.

### MODEL EVALUATION

In [ ]:
y_pred = model.predict(X_test)

print("************* Classification Report ************")
class_report = classification_report(y_test, y_pred, target_names = ['normal', 'droput', 'spike', 'flatline'])
print(class_report)

print("************* Confusion Matrix ************")
fix, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels = ['normal', 'droput', 'spike', 'flatline'], ax=ax, colorbar=False, cmap='Blues') # ax=ax 
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()  

In [ ]:
# print confusion matrix numbers
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['normal', 'droput', 'spike', 'flatline'], columns=['normal', 'droput', 'spike', 'flatline'])
print("Confusion Matrix:\n", cm_df)

### FEATURE IMPORTANCE

In [ ]:
importances = pd.Series(model.feature_importances_, index = X.columns).sort_values()
importances.tail(10).plot(kind = 'barh', figsize = (8,5), color = 'SteelBlue')
plt.title("Top 10 Feature Importances")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

### SAVE MODEL

In [ ]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/xgb_anomaly_classifier.pkl")

# also save the feature columns so that we can use them during inference time to ensure the same order of features is used.
joblib.dump(X.columns.tolist(), "../models/feature_columns.pkl")

print("Model saved to ../models")